# Lab | Data Aggregation and Filtering

In this challenge, we will continue to work with customer data from an insurance company. We will use the dataset called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by first performing data cleaning, formatting, and structuring.

1. Create a new DataFrame that only includes customers who:
   - have a **low total_claim_amount** (e.g., below $1,000),
   - have a response "Yes" to the last marketing campaign.

2. Using the original Dataframe, analyze:
   - the average `monthly_premium` and/or customer lifetime value by `policy_type` and `gender` for customers who responded "Yes", and
   - compare these insights to `total_claim_amount` patterns, and discuss which segments appear most profitable or low-risk for the company.

3. Analyze the total number of customers who have policies in each state, and then filter the results to only include states where there are more than 500 customers.

4. Find the maximum, minimum, and median customer lifetime value by education level and gender. Write your conclusions.

## Bonus

5. The marketing team wants to analyze the number of policies sold by state and month. Present the data in a table where the months are arranged as columns and the states are arranged as rows.

6.  Display a new DataFrame that contains the number of policies sold by month, by state, for the top 3 states with the highest number of policies sold.

*Hint:*
- *To accomplish this, you will first need to group the data by state and month, then count the number of policies sold for each group. Afterwards, you will need to sort the data by the count of policies sold in descending order.*
- *Next, you will select the top 3 states with the highest number of policies sold.*
- *Finally, you will create a new DataFrame that contains the number of policies sold by month for each of the top 3 states.*

7. The marketing team wants to analyze the effect of different marketing channels on the customer response rate.

Hint: You can use melt to unpivot the data and create a table that shows the customer response rate (those who responded "Yes") by marketing channel.

External Resources for Data Filtering: https://towardsdatascience.com/filtering-data-frames-in-pandas-b570b1f834b9

In [2]:
import pandas as pd

# Load the dataset
url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv"
df = pd.read_csv(url)

# 1. Filter for low total claim amount and 'Yes' responses
low_claim_yes_response_df = df[(df['Total Claim Amount'] < 1000) & (df['Response'] == 'Yes')]
print("1. Customers with low total claim amount and 'Yes' to response:")
print(low_claim_yes_response_df.head())

# 2. Average metrics by policy type and gender for 'Yes' responses
avg_metrics_df = df[df['Response'] == 'Yes'].groupby(['Policy Type', 'Gender']).agg({
    'Monthly Premium Auto': 'mean',
    'Customer Lifetime Value': 'mean',
    'Total Claim Amount': 'mean'
}).reset_index()
print("\n2. Average metrics by policy type and gender for 'Yes' responses:")
print(avg_metrics_df)

# 3. Count customers per state (only where more than 500 customers)
customers_by_state = df['State'].value_counts().reset_index()
customers_by_state.columns = ['State', 'Customer Count']
filtered_states = customers_by_state[customers_by_state['Customer Count'] > 500]
print("\n3. States with more than 500 customers:")
print(filtered_states)

# 4. Lifetime value stats by education and gender
clv_stats = df.groupby(['Education', 'Gender'])['Customer Lifetime Value'].agg(['max', 'min', 'median'])
print("\n4. Customer lifetime value stats by education and gender:")
print(clv_stats)

# 5. Policies sold by state and month
df['Effective To Date'] = pd.to_datetime(df['Effective To Date'])
df['Month'] = df['Effective To Date'].dt.strftime('%B')
policies_by_state_month = df.groupby(['State', 'Month']).size().unstack(fill_value=0)
print("\n5. Policies sold by state and month:")
print(policies_by_state_month)

# 6. Find the top 3 states with the most policies sold
policies_by_state_month_count = df.groupby(['State', 'Month']).size().reset_index(name='Policy Count')
top_states_by_policy = policies_by_state_month_count.groupby('State')['Policy Count'].sum().nlargest(3).index
top_policies_by_state_month = policies_by_state_month_count[policies_by_state_month_count['State'].isin(top_states_by_policy)]
print("\n6. Number of policies sold by month for the top 3 states:")
print(top_policies_by_state_month)

1. Customers with low total claim amount and 'Yes' to response:
    Unnamed: 0 Customer       State  Customer Lifetime Value Response  \
3            3  XL78013      Oregon             22332.439460      Yes   
8            8  FM55990  California              5989.773931      Yes   
15          15  CW49887  California              4626.801093      Yes   
19          19  NJ54277  California              3746.751625      Yes   
27          27  MQ68407      Oregon              4376.363592      Yes   

    Coverage Education Effective To Date EmploymentStatus Gender  ...  \
3   Extended   College           1/11/11         Employed      M  ...   
8    Premium   College           1/19/11         Employed      M  ...   
15     Basic    Master           1/16/11         Employed      F  ...   
19  Extended   College           2/26/11         Employed      F  ...   
27   Premium  Bachelor           2/28/11         Employed      F  ...   

    Number of Open Complaints Number of Policies     Polic

C:\Users\LAP-MPC\AppData\Local\Temp\ipykernel_11684\4166574666.py:34: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Effective To Date'] = pd.to_datetime(df['Effective To Date'])



5. Policies sold by state and month:
Month       February  January
State                        
Arizona          929     1008
California      1634     1918
Nevada           442      551
Oregon          1344     1565
Washington       425      463

6. Number of policies sold by month for the top 3 states:
        State     Month  Policy Count
0     Arizona  February           929
1     Arizona   January          1008
2  California  February          1634
3  California   January          1918
6      Oregon  February          1344
7      Oregon   January          1565
